# 02 — EDA: distribuzioni e confronto QCD vs EJ

**Obiettivo:** disegnare le distribuzioni delle feature di jet e di traccia, sovrapponendo QCD (background) ed Emerging Jets (signal). Validare visivamente le scelte di cleaning che abbiamo deciso nel notebook 01.

**Strategia di caricamento:** carichiamo **50.000 jet per file** (background + signal = 100k totali) per avere statistiche robuste senza sforare i 16 GB di RAM.

**Cosa NON fa** (lo faremo dopo):
- non applica il cleaning vero (questo va in `03_cleaning.ipynb`)
- non scrive il dataset clean su disco
- non costruisce il `Dataset` PyTorch (notebook 04)

**Sezioni:**
1. Setup e caricamento dati
2. Distribuzione del numero di tracce per jet
3. Feature di jet (alto livello)
4. Feature di traccia (basso livello)
5. Correlazioni e scatter plot 2D
6. Diagnostiche speciali (sentinel -999, overflow, outlier)
7. Salvataggio figure

## 0. Setup

Carichiamo le librerie e definiamo lo stile dei plot.

In [ ]:
import os
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)  # silenzia warning di overflow durante i plot

# Stile dei plot
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

# Colori standard (sempre gli stessi nel resto del notebook)
COLOR_QCD = '#1f77b4'   # blu
COLOR_EJ  = '#d62728'   # rosso

print('Setup completato.')

In [ ]:
# --- CONFIG ---------------------------------------------------
PATH_BKG = 'pp_output_test_background.h5'  # QCD
PATH_SIG = 'pp_output_test_signal.h5'      # Emerging Jets

N_SAMPLE = 50_000  # quanti jet caricare per file

FIGS_DIR = Path('figs')
FIGS_DIR.mkdir(exist_ok=True)

assert os.path.exists(PATH_BKG), f'File non trovato: {PATH_BKG}'
assert os.path.exists(PATH_SIG), f'File non trovato: {PATH_SIG}'
print(f'Background: {PATH_BKG} ({os.path.getsize(PATH_BKG)/1e9:.2f} GB)')
print(f'Signal:     {PATH_SIG} ({os.path.getsize(PATH_SIG)/1e9:.2f} GB)')
print(f'Sample per file: {N_SAMPLE:,}')
print(f'Le figure verranno salvate in: {FIGS_DIR.resolve()}')

## 1. Caricamento dei dati

Carichiamo i primi `N_SAMPLE` jet (e le rispettive tracce) da entrambi i file. Convertiamo a `float32` ovunque per evitare gli overflow di `float16` che abbiamo visto nel notebook 01.

In [ ]:
def load_jets_and_tracks(path, n_sample):
    """Carica jets e tracks da un file H5, primi n_sample esempi."""
    with h5py.File(path, 'r') as f:
        n = min(n_sample, f['jets'].shape[0])
        jets = f['jets'][:n]
        tracks = f['tracks'][:n]
    return jets, tracks

print('Caricamento background QCD...')
jets_bkg, tracks_bkg = load_jets_and_tracks(PATH_BKG, N_SAMPLE)
print(f'  jets:   {jets_bkg.shape}')
print(f'  tracks: {tracks_bkg.shape}')

print('\nCaricamento signal EJ...')
jets_sig, tracks_sig = load_jets_and_tracks(PATH_SIG, N_SAMPLE)
print(f'  jets:   {jets_sig.shape}')
print(f'  tracks: {tracks_sig.shape}')

# Memoria usata (approssimativa)
mem_mb = (jets_bkg.nbytes + tracks_bkg.nbytes + jets_sig.nbytes + tracks_sig.nbytes) / 1e6
print(f'\nMemoria totale usata: ~{mem_mb:.0f} MB')

In [ ]:
# Helper: estrae una feature da uno structured array convertendola a float32
# (fondamentale per evitare gli overflow di float16 sui valori grandi)
def get_feat(structured_arr, name):
    """Estrae una feature come array float32."""
    return structured_arr[name].astype(np.float32)

# Maschere booleane delle tracce valide (True dove la traccia esiste)
mask_bkg = tracks_bkg['valid']
mask_sig = tracks_sig['valid']

print(f'Tracce valide BKG: {mask_bkg.sum():,} / {mask_bkg.size:,} '
      f'({mask_bkg.mean():.1%})')
print(f'Tracce valide SIG: {mask_sig.sum():,} / {mask_sig.size:,} '
      f'({mask_sig.mean():.1%})')

## 2. Distribuzione del numero di tracce per jet

Il numero di tracce per jet e una delle feature piu **discriminanti** tra QCD ed EJ: gli EJ hanno tipicamente piu tracce perche provengono da molti decay vertex displaced.

Questo plot ci dice anche il **padding ottimale** per il modello: se il P99 e ~125, possiamo tagliare a 128 senza perdere informazione.

In [ ]:
n_tracks_bkg = mask_bkg.sum(axis=1)
n_tracks_sig = mask_sig.sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

bins = np.arange(0, 201, 5)
for ax, scale in zip(axes, ['linear', 'log']):
    ax.hist(n_tracks_bkg, bins=bins, alpha=0.6, density=True,
            color=COLOR_QCD, label=f'QCD (n={len(n_tracks_bkg):,})')
    ax.hist(n_tracks_sig, bins=bins, alpha=0.6, density=True,
            color=COLOR_EJ, label=f'EJ (n={len(n_tracks_sig):,})')
    ax.set_xlabel('numero di tracce valide per jet')
    ax.set_ylabel('densita')
    ax.set_yscale(scale)
    ax.set_title(f'N tracce per jet ({scale})')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGS_DIR / 'n_tracks_per_jet.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nStatistiche QCD:  mean={n_tracks_bkg.mean():.1f}  median={np.median(n_tracks_bkg):.0f}  P99={np.percentile(n_tracks_bkg, 99):.0f}')
print(f'Statistiche EJ:   mean={n_tracks_sig.mean():.1f}  median={np.median(n_tracks_sig):.0f}  P99={np.percentile(n_tracks_sig, 99):.0f}')

## 3. Feature di jet (alto livello)

Disegniamo gli istogrammi sovrapposti QCD vs EJ delle feature di jet piu importanti. Definiamo una funzione `compare_hist` che ci permette di farlo in modo uniforme.

In [ ]:
def compare_hist(values_bkg, values_sig, name, bins=60, log_y=False,
                 clip=None, save=True, show=True):
    """Disegna istogrammi sovrapposti QCD vs EJ per una feature.
    
    Args:
        values_bkg, values_sig: array delle feature (gia filtrati e float32)
        name: nome della feature (per titolo e file)
        bins: numero di bin o array di bin edges
        log_y: scala log sull'asse y
        clip: tupla (low, high) per clippare gli outlier prima del plot
    """
    bkg = values_bkg[np.isfinite(values_bkg)]
    sig = values_sig[np.isfinite(values_sig)]
    
    if clip is not None:
        bkg = np.clip(bkg, clip[0], clip[1])
        sig = np.clip(sig, clip[0], clip[1])
    
    # range comune basato sui percentili (per non farsi rovinare dagli outlier)
    if isinstance(bins, int):
        combined = np.concatenate([bkg, sig])
        lo, hi = np.percentile(combined, [0.5, 99.5])
        if lo == hi:
            lo, hi = combined.min(), combined.max() + 1
        bins = np.linspace(lo, hi, bins)
    
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(bkg, bins=bins, alpha=0.6, density=True,
            color=COLOR_QCD, label=f'QCD (n={len(bkg):,})')
    ax.hist(sig, bins=bins, alpha=0.6, density=True,
            color=COLOR_EJ, label=f'EJ  (n={len(sig):,})')
    ax.set_xlabel(name)
    ax.set_ylabel('densita' + (' (log)' if log_y else ''))
    if log_y:
        ax.set_yscale('log')
    ax.set_title(f'Distribuzione {name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    if save:
        suffix = '_log' if log_y else ''
        fig.savefig(FIGS_DIR / f'jet_{name}{suffix}.png', dpi=120, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)

print('Funzione compare_hist pronta.')

### 3.1 Feature cinematiche (pt, eta, mass, energy)

Aspettative: pt ed energy molto simili tra QCD ed EJ (entrambi sono jet di alto pT). Mass piu spostata verso valori piu alti per EJ (jet con sottostruttura).

In [ ]:
for feat in ['pt', 'eta', 'mass', 'energy']:
    compare_hist(get_feat(jets_bkg, feat), get_feat(jets_sig, feat), feat, bins=60)

### 3.2 Feature 'displaced' di ATLAS

**Aspettative chiave:**
- `displacedPtFraction` molto piu alta per EJ (e proprio la PTF dell'articolo cut-based!). Attenzione: ha max 5.6 → clippiamo a 1.5
- `salt_pdisp` (classificatore SALT gia pre-trained): ~1 per EJ, ~0 per QCD
- `timing_clusterSoftDrop`: gli EJ hanno tracce displaced che possono avere timing diverso

In [ ]:
compare_hist(get_feat(jets_bkg, 'displacedPtFraction'),
             get_feat(jets_sig, 'displacedPtFraction'),
             'displacedPtFraction', bins=60, clip=(0, 1.5))

compare_hist(get_feat(jets_bkg, 'salt_pdisp'),
             get_feat(jets_sig, 'salt_pdisp'),
             'salt_pdisp', bins=60, log_y=True)

compare_hist(get_feat(jets_bkg, 'timing_clusterSoftDrop'),
             get_feat(jets_sig, 'timing_clusterSoftDrop'),
             'timing_clusterSoftDrop', bins=60, clip=(-5, 5))

### 3.3 N-subjettiness (Tau) e energy correlations (ECF)

Per Tau21 e Tau32: ricordiamoci che hanno il sentinel -999 (jet senza sottostruttura). Lo filtriamo prima del plot.

In [ ]:
for feat in ['Tau1_clusterSoftDrop', 'Tau2_clusterSoftDrop',
             'Tau3_clusterSoftDrop', 'Tau4_clusterSoftDrop']:
    compare_hist(get_feat(jets_bkg, feat), get_feat(jets_sig, feat), feat, bins=60)

In [ ]:
# Tau21 e Tau32 hanno sentinel -999: filtriamo prima del plot
for feat in ['Tau21_clusterSoftDrop', 'Tau32_clusterSoftDrop']:
    bkg = get_feat(jets_bkg, feat)
    sig = get_feat(jets_sig, feat)
    bkg_clean = bkg[bkg > -100]   # tutto cio che e > -100 e fisico
    sig_clean = sig[sig > -100]
    print(f'{feat}: sentinel in QCD = {(bkg <= -100).mean():.1%}, in EJ = {(sig <= -100).mean():.1%}')
    compare_hist(bkg_clean, sig_clean, feat, bins=60)

In [ ]:
# ECF: hanno range enorme, usiamo log
for feat in ['ECF1_clusterSoftDrop', 'ECF2_clusterSoftDrop',
             'ECF3_clusterSoftDrop', 'ECF4_clusterSoftDrop']:
    bkg = get_feat(jets_bkg, feat)
    sig = get_feat(jets_sig, feat)
    # log1p (= log(1+x)) per evitare problemi con 0 e con i valori giganteschi
    bkg_log = np.log1p(np.clip(bkg, 0, None))
    sig_log = np.log1p(np.clip(sig, 0, None))
    compare_hist(bkg_log, sig_log, f'log1p({feat})', bins=60)

In [ ]:
# C2, D2: hanno sentinel -999
for feat in ['C2_clusterSoftDrop', 'D2_clusterSoftDrop']:
    bkg = get_feat(jets_bkg, feat)
    sig = get_feat(jets_sig, feat)
    bkg_clean = bkg[bkg > -100]
    sig_clean = sig[sig > -100]
    print(f'{feat}: sentinel in QCD = {(bkg <= -100).mean():.1%}, in EJ = {(sig <= -100).mean():.1%}')
    compare_hist(bkg_clean, sig_clean, feat, bins=60)

### 3.4 Pile-up e altre feature di contesto

In [ ]:
for feat in ['nPrimaryVertices', 'averageInteractionsPerCrossing']:
    compare_hist(get_feat(jets_bkg, feat), get_feat(jets_sig, feat), feat, bins=60)

## 4. Feature di traccia

Per le tracce dobbiamo lavorare **solo sulle tracce valide** (usando la maschera). Definiamo una funzione che estrae una feature da tutte le tracce valide.

In [ ]:
def get_track_feat(tracks, mask, name):
    """Estrae una feature di tracce, solo sulle tracce valide."""
    arr = tracks[name][mask].astype(np.float32)
    return arr[np.isfinite(arr)]

def compare_hist_track(name, bins=60, log_y=False, clip=None, transform=None):
    """Helper per istogrammi di tracce."""
    bkg = get_track_feat(tracks_bkg, mask_bkg, name)
    sig = get_track_feat(tracks_sig, mask_sig, name)
    if transform is not None:
        bkg = transform(bkg)
        sig = transform(sig)
        name_disp = f'{transform.__name__}({name})' if hasattr(transform, '__name__') else name
    else:
        name_disp = name
    
    if clip is not None:
        bkg = np.clip(bkg, clip[0], clip[1])
        sig = np.clip(sig, clip[0], clip[1])
    
    if isinstance(bins, int):
        combined = np.concatenate([bkg, sig])
        lo, hi = np.percentile(combined, [0.5, 99.5])
        if lo == hi:
            lo, hi = combined.min(), combined.max() + 1
        bins = np.linspace(lo, hi, bins)
    
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(bkg, bins=bins, alpha=0.6, density=True, color=COLOR_QCD,
            label=f'QCD (n={len(bkg):,})')
    ax.hist(sig, bins=bins, alpha=0.6, density=True, color=COLOR_EJ,
            label=f'EJ  (n={len(sig):,})')
    ax.set_xlabel(name_disp)
    ax.set_ylabel('densita')
    if log_y:
        ax.set_yscale('log')
    ax.set_title(f'Tracce - distribuzione {name_disp}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.savefig(FIGS_DIR / f'track_{name}.png', dpi=120, bbox_inches='tight')
    plt.show()

print('Funzioni helper pronte.')

### 4.1 Parametri di impatto (i piu importanti!)

**Aspettativa fortissima:** EJ ha tracce con `d0` e `z0` displaced (lontane dal vertice primario), QCD ha tracce prompt. Sono **le feature numero 1 per discriminare**.

In [ ]:
compare_hist_track('d0_log1p', bins=60)
compare_hist_track('d0_log1p', bins=60, log_y=True)
compare_hist_track('IP3D_signed_d0_significance', bins=80, clip=(-100, 100))

In [ ]:
# d0 raw clippato (per visualizzare)
compare_hist_track('d0', bins=60, clip=(-50, 50))
compare_hist_track('z0SinTheta', bins=60, clip=(-200, 200))

### 4.2 Hits nel tracker (PARTICOLARMENTE INFORMATIVI!)

Le tracce displaced **partono dentro al rivelatore**, quindi mancano gli hit nel layer piu interno (IBL). Questa e una delle feature piu predittive.

In [ ]:
# Hit counts: sono discrete (0, 1, 2, 3, ...) -> usiamo bin interi
for feat, max_val in [('numberOfInnermostPixelLayerHits', 5),
                       ('numberOfPixelHits', 10),
                       ('numberOfPixelHoles', 5),
                       ('numberOfSCTHits', 15),
                       ('numberOfSCTHoles', 5),
                       ('numberOfPixelSharedHits', 5),
                       ('numberOfSCTSharedHits', 5)]:
    compare_hist_track(feat, bins=np.arange(0, max_val+1))

### 4.3 Cinematica e momento della traccia

In [ ]:
compare_hist_track('pt_log1p', bins=60)
compare_hist_track('qOverP', bins=60, clip=(-0.001, 0.001))
compare_hist_track('theta', bins=60)
compare_hist_track('deta', bins=60, clip=(-1, 1))
compare_hist_track('dphi', bins=60, clip=(-1, 1))
compare_hist_track('dr', bins=60, clip=(0, 1))

### 4.4 Qualita e raggio della prima hit

`radiusOfFirstHit` e estremamente informativo: tracce displaced hanno primo hit a raggio maggiore.

In [ ]:
compare_hist_track('radiusOfFirstHit', bins=60, clip=(0, 400))
compare_hist_track('chiSquared', bins=60, clip=(0, 100))
compare_hist_track('numberDoF', bins=60, clip=(0, 50))

## 5. Correlazioni tra feature di jet

Vediamo quali feature di jet sono correlate. Calcoliamo la matrice di correlazione sul background (perche e quello su cui addestreremo il modello unsupervised).

In [ ]:
# Selezioniamo solo le feature 'usabili' (no sentinel, no overflow)
FEATS_FOR_CORR = [
    'pt', 'eta', 'mass', 'energy', 'displacedPtFraction',
    'salt_pdisp', 'Qw_clusterSoftDrop',
    'Tau1_clusterSoftDrop', 'Tau2_clusterSoftDrop',
    'Tau3_clusterSoftDrop', 'Tau4_clusterSoftDrop',
    'Split12_clusterSoftDrop', 'Split23_clusterSoftDrop',
    'timing_clusterSoftDrop', 'nPrimaryVertices',
]

# Costruisco un array (n_jets, n_features) per il QCD
data_corr = np.stack([get_feat(jets_bkg, f) for f in FEATS_FOR_CORR], axis=1)
# Rimuovo righe con NaN/Inf
ok = np.all(np.isfinite(data_corr), axis=1)
data_corr = data_corr[ok]
print(f'Sample valido: {data_corr.shape[0]:,} jets')

corr = np.corrcoef(data_corr.T)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATS_FOR_CORR)))
ax.set_yticks(range(len(FEATS_FOR_CORR)))
ax.set_xticklabels(FEATS_FOR_CORR, rotation=90, fontsize=8)
ax.set_yticklabels(FEATS_FOR_CORR, fontsize=8)
for i in range(len(FEATS_FOR_CORR)):
    for j in range(len(FEATS_FOR_CORR)):
        ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center',
                fontsize=6, color='white' if abs(corr[i,j]) > 0.5 else 'black')
ax.set_title('Matrice di correlazione - feature di jet (QCD)')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'corr_matrix_jets.png', dpi=120, bbox_inches='tight')
plt.show()

### 5.1 Scatter plot 2D di coppie chiave

Vediamo `displacedPtFraction` vs `salt_pdisp`: sono i due predittori 'ufficiali' di ATLAS. Devono separare bene QCD da EJ.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(get_feat(jets_bkg, 'displacedPtFraction'),
           get_feat(jets_bkg, 'salt_pdisp'),
           s=2, alpha=0.3, color=COLOR_QCD, label='QCD')
ax.scatter(get_feat(jets_sig, 'displacedPtFraction'),
           get_feat(jets_sig, 'salt_pdisp'),
           s=2, alpha=0.3, color=COLOR_EJ, label='EJ')
ax.set_xlim(0, 1.5)
ax.set_xlabel('displacedPtFraction')
ax.set_ylabel('salt_pdisp')
ax.set_title('displacedPtFraction vs salt_pdisp')
ax.legend(markerscale=3)
ax.grid(True, alpha=0.3)
fig.savefig(FIGS_DIR / 'scatter_PTF_vs_salt.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Diagnostiche speciali

Verifichiamo quantitativamente i red flag che avevamo identificato nel notebook 01.

### 6.1 Sentinel -999 (frequenza)

Quanto spesso compare il sentinel -999 nelle feature problematiche?

In [ ]:
SENTINEL_FEATS = ['C2_clusterSoftDrop', 'D2_clusterSoftDrop',
                  'Tau21_clusterSoftDrop', 'Tau32_clusterSoftDrop']

print(f"{'feature':30s} {'QCD':>10s} {'EJ':>10s}")
print('-' * 55)
for feat in SENTINEL_FEATS:
    bkg = get_feat(jets_bkg, feat)
    sig = get_feat(jets_sig, feat)
    p_bkg = (bkg <= -100).mean()
    p_sig = (sig <= -100).mean()
    print(f'{feat:30s} {p_bkg:9.1%}  {p_sig:9.1%}')

### 6.2 Outlier estremi su feature di traccia

Quante tracce hanno valori 'patologici' (pt > 100 GeV, |d0| > 200 mm, ecc.)?

In [ ]:
def check_outliers(name, condition_fn, condition_str):
    arr_bkg = get_track_feat(tracks_bkg, mask_bkg, name)
    arr_sig = get_track_feat(tracks_sig, mask_sig, name)
    p_bkg = condition_fn(arr_bkg).mean()
    p_sig = condition_fn(arr_sig).mean()
    print(f'{name:35s} {condition_str:20s} QCD={p_bkg:8.2%}  EJ={p_sig:8.2%}')

check_outliers('pt', lambda x: x > 1e5, 'pt > 100 GeV')
check_outliers('pt', lambda x: x > 1e6, 'pt > 1 TeV')
check_outliers('d0', lambda x: np.abs(x) > 200, '|d0| > 200 mm')
check_outliers('z0SinTheta', lambda x: np.abs(x) > 300, '|z0sinT| > 300 mm')
check_outliers('IP3D_signed_d0_significance', lambda x: np.abs(x) > 100, '|IP3D sig| > 100')
check_outliers('chiSquared', lambda x: x > 100, 'chi2 > 100')

### 6.3 Bilanciamento label nei due file

Verifichiamo che background contenga effettivamente solo QCD (isDisplaced=0) e signal solo EJ (isDisplaced=1).

In [ ]:
for label_name, jets_arr, file_name in [('background', jets_bkg, PATH_BKG),
                                          ('signal', jets_sig, PATH_SIG)]:
    isd = jets_arr['isDisplaced']
    vals, cnt = np.unique(isd, return_counts=True)
    print(f'\n{file_name} (label = {label_name}):')
    for v, c in zip(vals, cnt):
        print(f'  isDisplaced={v}: {c:,}  ({c/len(isd):.1%})')

## 7. Conclusioni

Riepilogo di cosa abbiamo imparato e che servira per il notebook 03 (cleaning vero):

**Numero di tracce:**
- EJ hanno (mediamente) **piu tracce** di QCD → e una feature discriminante naturale
- P99 ~ 125 → **padding a 128** e sensato

**Feature di jet piu discriminanti:**
- `displacedPtFraction` (e la PTF dell'articolo)
- `salt_pdisp` (gia un classificatore)
- `mass`, `Tau21/Tau32` (sottostruttura)

**Feature di traccia piu discriminanti:**
- `d0`, `d0_log1p`, `z0SinTheta` (tracce displaced)
- `IP3D_signed_d0_significance`
- `numberOfInnermostPixelLayerHits` (assenza di hit IBL!)
- `radiusOfFirstHit`

**Problemi confermati dall'EDA:**
- Sentinel -999 in C2, D2, Tau21, Tau32: presente in una frazione non trascurabile di jet
- Outlier estremi: pt traccia > 100 GeV, |d0| > 200 mm sono rari ma esistono
- ECF4 va trattata con log per evitare overflow
- `displacedPtFraction` ha valori > 1 (bug) → da clippare

**Prossimo passo:** notebook `03_cleaning.ipynb` che implementa tutte queste regole come codice e produce il dataset 'clean' da dare al modello.

In [ ]:
# Liberiamo la memoria (utile se proseguiamo a lavorare nel notebook)
del jets_bkg, tracks_bkg, jets_sig, tracks_sig, mask_bkg, mask_sig
import gc
gc.collect()
print('Memoria liberata. Notebook completato.')